# Model Selection — AIC vs BIC vs State Count

Deep-dive into how the engine chooses the number of hidden states.
We plot the full AIC/BIC curves and inspect what changes at each candidate `n_states`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as sp

from rde.config import load_config
from rde.data import YFinanceSource
from rde.features import FeaturePipeline, LogReturns, RollingVolatility, SmoothedReturns
from rde.models import select_n_states
from rde.models.hmm import train_hmm

In [ ]:
cfg = load_config(Path('../configs/btc.yaml'))
source = YFinanceSource(cache_dir=Path('../results/cache'))
df = source.load(cfg.asset.symbol, cfg.asset.period, cfg.asset.interval)
pipeline = FeaturePipeline([LogReturns(), RollingVolatility(24), SmoothedReturns(12)])
df_feat = pipeline.transform(df)
X = df_feat[pipeline.output_columns].values
print(f'{len(X)} observations, {X.shape[1]} features')

## AIC / BIC curves across candidate state counts

For each `n_states` in the candidate set we run the full multi-restart training
and record the information criteria. The engine selects the minimum.

In [ ]:
_, scores_df = select_n_states(
    X,
    candidate_states=cfg.model.candidate_states,
    criterion=cfg.selection.criterion,
    n_restarts=cfg.model.n_restarts,
    covariance_type=cfg.model.covariance_type,
    n_iter=cfg.model.n_iter,
    seed_base=cfg.model.seed_base,
    feature_names=pipeline.output_columns,
)
scores_df

In [ ]:
fig = sp.make_subplots(rows=1, cols=2, subplot_titles=['AIC', 'BIC'])

for col, metric in [(1, 'aic'), (2, 'bic')]:
    fig.add_trace(
        go.Scatter(
            x=scores_df['n_states'],
            y=scores_df[metric],
            mode='lines+markers',
            name=metric.upper(),
            marker=dict(size=8),
        ),
        row=1, col=col,
    )
    best = scores_df.loc[scores_df[metric].idxmin()]
    fig.add_vline(x=best['n_states'], line_dash='dot', line_color='red', row=1, col=col)

fig.update_layout(title='Model selection: AIC and BIC by number of states', height=400)
fig.show()

## Restart score distribution for the selected model

Because HMM training is non-convex we run multiple restarts and keep the best.
Here we inspect how spread the restart log-likelihoods are — tight distributions
mean the optimiser reliably finds the same solution.

In [ ]:
best_n = int(scores_df.loc[scores_df[cfg.selection.criterion].idxmin(), 'n_states'])
print(f'Best n_states = {best_n}')

fitted = train_hmm(
    X, best_n,
    n_restarts=cfg.model.n_restarts,
    covariance_type=cfg.model.covariance_type,
    n_iter=cfg.model.n_iter,
    seed_base=cfg.model.seed_base,
    feature_names=pipeline.output_columns,
)

fig = go.Figure(go.Bar(
    x=list(range(len(fitted.all_restart_scores))),
    y=fitted.all_restart_scores,
    marker_color=['#d62728' if s == max(fitted.all_restart_scores) else '#1f77b4'
                  for s in fitted.all_restart_scores],
))
fig.update_layout(
    title=f'Log-likelihood per restart (n_states={best_n}, red=winner)',
    xaxis_title='Restart index',
    yaxis_title='Log-likelihood',
    height=350,
)
fig.show()